# Free-Lunch AI Demo 🍱

One YAML, one entrypoint, multiple free LLM providers with automatic fallback.

**Table of Contents**
1. [Installation & Setup](#1-installation--setup)
2. [Quick Start — LangChain Router](#2-quick-start--langchain-router)
3. [Quick Start — Light Router](#3-quick-start--light-router)
4. [Custom YAML Menu](#4-custom-yaml-menu)
5. [Utilities](#5-utilities)

---
## 1. Installation & Setup
<a id='1-installation--setup'></a>

In [ ]:
# With LangChain support
!pip install "free-lunch-ai[langchain] @ git+https://github.com/tctsung/free-lunch-ai.git" --force-reinstall -q

# Or light only (no LangChain dependency):
# !pip install git+https://github.com/tctsung/free-lunch-ai.git --force-reinstall -q

In [ ]:
import os

# Option A: export keys directly (uncomment and fill in)
# default:: provider requires no key — always available
# os.environ["GROQ_API_KEY"] = "your-key"          # https://console.groq.com/keys
# os.environ["GOOGLE_API_KEY"] = "your-key"         # https://aistudio.google.com/app/api-keys
# os.environ["OPENROUTER_API_KEY"] = "your-key"     # https://openrouter.ai/settings/keys
# os.environ["OLLAMA_API_KEY"] = "your-key"         # https://ollama.com/settings/keys

# Option B: use a .env file (place in working directory)
# The Menu class loads .env automatically, no extra code needed.

---
## 2. Quick Start — LangChain Router
<a id='2-quick-start--langchain-router'></a>

Requires `pip install free-lunch-ai[langchain]` or `[all]`.

Returns standard LangChain `AIMessage` — works with `.bind_tools()`, agents, chains.

In [ ]:
from free_lunch import Menu

menu = Menu()  # auto-detects langchain vs light; override with Menu(router_type="light")

# Fast preset
fast_llm = menu.fast()
response = fast_llm.invoke("Explain no free lunch theorem in one sentence.")

print(response.content)
print("Model used:", response.response_metadata["model_id"])

In [ ]:
# Think preset
think_llm = menu.think()
response = think_llm.invoke("Prove that the square root of 2 is irrational.")

print("Model used:", response.response_metadata["model_id"])
print(response.content)

In [ ]:
# Agent preset (Groq Compound — built-in web search, code exec)
agent_llm = menu.agent()
response = agent_llm.invoke("What is the weather in NYC right now?")

print("Model used:", response.response_metadata["model_id"])
print(response.content)

---
## 3. Quick Start — Light Router
<a id='3-quick-start--light-router'></a>

No LangChain dependency. Uses raw `httpx` with OpenAI-compatible endpoints.

Returns a plain dict: `{"text": "...", "model_id": "..."}`.

In [ ]:
from free_lunch import Menu

# Use type: light in YAML, or create a LightRouter directly
from free_lunch.light_router import LightRouter

router = LightRouter(
    func_name="fast",
    models=[
        {"id": "default::openai"},              # no API key required
        {"id": "groq::llama-3.1-8b-instant"},
        {"id": "google::gemini-2.5-flash"},
        {"id": "openrouter::qwen/qwen3-4b:free"},
    ],
    timeout=30,
    global_timeout=180,
)

result = router.invoke("Explain no free lunch theorem in one sentence.")
print(result)
# {"text": "...", "model_id": "groq::llama-3.1-8b-instant"}

In [ ]:
# Or use via YAML with type: light
%%writefile my_light_menu.yaml
fast:
  type: light
  timeout: 30
  models:
    - id: default::openai-fast    # no API key required
    - id: groq::llama-3.1-8b-instant
    - id: google::gemini-2.5-flash
    - id: openrouter::qwen/qwen3-4b:free

In [ ]:
menu = Menu(yaml_path="my_light_menu.yaml")
fast = menu.fast()

result = fast.invoke("What is 2 + 2?")
print(result)

# Also accepts message list format:
result = fast.invoke([{"role": "user", "content": "What is 2 + 2?"}])
print(result)

In [ ]:
import os
os.remove("my_light_menu.yaml")

---
## 4. Custom YAML Menu
<a id='4-custom-yaml-menu'></a>

Define your own profiles. Mix `langchain` and `light` types in the same file.

In [ ]:
%%writefile my_menu.yaml
# LangChain router — works with agents, .bind_tools(), chains
fast:
  type: langchain
  timeout: 30
  global_timeout: 180
  models:
    - id: default::openai           # no API key required
    - id: google::gemini-2.5-flash
    - id: groq::llama-3.1-8b-instant
    - id: openrouter::qwen/qwen3-4b:free

# Light router — no LangChain, returns plain dict
story_teller:
  type: light
  timeout: 90
  global_timeout: 300
  models:
    - id: default::openai-fast      # no API key required
    - id: ollama::deepseek-v3.2:cloud
    - id: groq::openai/gpt-oss-120b
      params:
        reasoning_effort: high
    - id: openrouter::deepseek/deepseek-r1-0528:free

In [ ]:
my_menu = Menu(yaml_path="my_menu.yaml")

# LangChain router
fast_llm = my_menu.fast()
response = fast_llm.invoke("Summarize the theory of relativity in 2 sentences.")
print("Model used:", response.response_metadata["model_id"])
print(response.content)

In [ ]:
# Light router
story = my_menu.story_teller()
result = story.invoke("Write a short fable about a fox who learns to share.")
print(f"Model: {result['model_id']}")
print(result["text"])

In [ ]:
import os
os.remove("my_menu.yaml")

---
## 5. Utilities
<a id='5-utilities'></a>

### Built-in tools

Free-Lunch includes a few useful API-key-free utilities for everyday agent context: current time, web search, and URL fetching.

Naming convention:
- Plain names (`current_time`, `web_search`, `fetch_url`) are normal Python functions.
- `_tool` names (`current_time_tool`, `web_search_tool`, `fetch_url_tool`) are ready for LangChain / LangGraph `.bind_tools()` workflows.

This example keeps the full agent response in `raw_response`, then prints the final answer clearly.

In [ ]:
from free_lunch import Menu, content_blocks_dict
from free_lunch.tools import web_search_tool, fetch_url_tool
from langchain.agents import create_agent
from pprint import pprint
fast_llm = Menu().fast()
agent = create_agent(
    model=fast_llm,
    tools=[web_search_tool, fetch_url_tool],
)

raw_response = agent.invoke({
    "messages": [{
        "role": "user",
        "content": (
            "Which teams reached the NBA conference finals in the 2026 playoffs? "
            "Search the web and answer with the Eastern and Western Conference matchups."
        ),
    }]
})

result = content_blocks_dict(raw_response)
pprint(result)


{'model_id': 'google::gemini-2.5-flash',
 'raw_response': {'messages': [HumanMessage(content='Which teams reached the NBA conference finals in the 2026 playoffs? Search the web and answer with the Eastern and Western Conference matchups.', additional_kwargs={}, response_metadata={}, id='415f2017-5357-4f69-ab29-cd5b273c8761'),
                               AIMessage(content='', additional_kwargs={'reasoning_content': 'User asks: "Which teams reached the NBA conference finals in the 2026 playoffs? Search the web and answer with the Eastern and Western Conference matchups."\n\nWe need to find info about 2026 NBA playoffs conference finals. Likely not known; 2026 future. But maybe there\'s a simulation or projection? Could be a hypothetical? But user explicitly says search the web. Let\'s search.', 'tool_calls': [{'id': 'fc_3924a89b-8e32-4367-a728-acadda4a9ce5', 'function': {'arguments': '{"max_results":10,"query":"2026 NBA playoffs conference finals teams"}', 'name': 'web_search'}, 'type

### `content_blocks_dict` (optional, LangChain only)

Flattens a LangChain `AIMessage` or `create_agent` response dict into a plain dict. Handles thinking content:
- If provider returns a separate `reasoning` field → included as `reasoning`
- If `<think>` tags in content → extracted to `reasoning`, stripped from `text`, original kept in `raw_text`

Not needed for Light router — it already returns a dict.

In [ ]:
from free_lunch import content_blocks_dict

fast_llm = Menu().fast()
response = fast_llm.invoke("What is 2 + 2?")

result = content_blocks_dict(response)
print(result)
# {"text": "4", "model_id": "google::gemini-2.5-flash"}
# If thinking model: also has "reasoning" and "raw_text" keys